# News-Sentiment Fine-Tuning — Setup → Baseline → Iteration 2

**Goal:** load Financial PhraseBank, inspect it, fine-tune DistilBERT for 3-way
sentiment, and report accuracy **and macro-F1** (imbalanced classes).

Plan reference: `docs/news_sentiment_finetuning_plan.md`.

Order of operations:
1. Imports + environment check
2. Load dataset and inspect (schema, labels, class balance)
3. Load tokenizer + classification model; sanity-check forward pass
4. Tokenize the **full** train/val/test splits
5. Train 3 epochs with Hugging Face `Trainer`; track macro-F1; keep best val-F1 checkpoint

In [1]:
# If running on a fresh machine / Colab, uncomment:
# %pip install -q transformers datasets evaluate accelerate scikit-learn

import platform

import numpy as np
import pandas as pd
import torch
import transformers
import datasets

print("Python      :", platform.python_version())
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)

# DEVICE is selected (and benchmarked) in the next section, "1·0 Device check".

Python      : 3.11.15
torch       : 2.12.1
transformers: 5.12.1
datasets    : 5.0.0


## 1·0. Device check & GPU benchmark

Detect every available backend (CUDA / Apple MPS / CPU), select the best one as
`DEVICE`, then **confirm** the accelerator is actually used by timing the same
matrix-multiply workload on CPU vs `DEVICE`.

Timing notes:
- GPUs run **asynchronously**, so we call `torch.cuda.synchronize()` /
  `torch.mps.synchronize()` before stopping the timer — otherwise we'd measure the
  launch, not the compute.
- We run a **warm-up** first (kernel compilation / lazy init) so the timed run is fair.

In [2]:
import time

import torch

# 1) Which backends are available?
cuda_ok = torch.cuda.is_available()
mps_ok = getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available()
print("Available backends")
print(f"  CUDA (NVIDIA): {cuda_ok}")
print(f"  MPS  (Apple) : {mps_ok}")
print(f"  CPU          : True (always)")

# 2) Select the best device.
if cuda_ok:
    DEVICE = "cuda"
    print("\nUsing CUDA GPU:", torch.cuda.get_device_name(0))
elif mps_ok:
    DEVICE = "mps"
    print("\nUsing Apple MPS GPU (Metal).")
else:
    DEVICE = "cpu"
    print("\nNo GPU detected — falling back to CPU.")
print("DEVICE =", DEVICE)


def benchmark_matmul(device: str, n: int = 4096, iters: int = 10) -> float:
    """Time `iters` n×n matmuls on `device`; returns seconds (sync-corrected)."""
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)

    def _sync():
        if device == "cuda":
            torch.cuda.synchronize()
        elif device == "mps":
            torch.mps.synchronize()

    # Warm-up (kernel compile / lazy alloc) — not timed.
    for _ in range(3):
        _ = a @ b
    _sync()

    start = time.perf_counter()
    for _ in range(iters):
        _ = a @ b
    _sync()
    return time.perf_counter() - start


# 3) Confirm the accelerator is actually used — time CPU vs DEVICE.
print("\nBenchmark: 10× (4096×4096) matmul")
cpu_time = benchmark_matmul("cpu")
print(f"  CPU   : {cpu_time:.3f}s")

if DEVICE != "cpu":
    gpu_time = benchmark_matmul(DEVICE)
    print(f"  {DEVICE.upper():<5} : {gpu_time:.3f}s")
    if gpu_time > 0:
        print(f"  Speed-up: {cpu_time / gpu_time:.1f}× faster on {DEVICE.upper()}")

    # Hard confirmation that a tensor really lives on the accelerator.
    probe = torch.ones(1, device=DEVICE)
    print(f"\nConfirmed: sample tensor is on '{probe.device}'.")
else:
    print("  (no accelerator to compare against)")

Available backends
  CUDA (NVIDIA): False
  MPS  (Apple) : True
  CPU          : True (always)

Using Apple MPS GPU (Metal).
DEVICE = mps

Benchmark: 10× (4096×4096) matmul


  CPU   : 0.439s


  MPS   : 0.104s
  Speed-up: 4.2× faster on MPS

Confirmed: sample tensor is on 'mps:0'.


## 1. Load the dataset — Financial PhraseBank

~4.8k finance sentences labeled **negative / neutral / positive**.

> **Compatibility gotcha (good interview talking point):** the canonical
> `financial_phrasebank` repo ships as a *Python loading script*, and `datasets`
> v4/v5 **removed script support** (`RuntimeError: Dataset scripts are no longer
> supported`). `trust_remote_code=True` was also removed. The fix is to load a
> **script-free Parquet mirror**. We use `atrost/financial_phrasebank`, which has
> proper `ClassLabel` names and ready-made train/validation/test splits
> (3100 / 776 / 970 = 4846, matching the original `sentences_50agree` config).

In [3]:
from datasets import load_dataset, ClassLabel

# Script-free Parquet mirrors (the canonical script-based repo no longer loads on
# datasets v5 — see the note above).
PRIMARY_SOURCE = "atrost/financial_phrasebank"          # ClassLabel + train/val/test splits
FALLBACK_SOURCE = "warwickai/financial_phrasebank_mirror"  # single train split, int labels
LABEL_NAMES_FALLBACK = ["negative", "neutral", "positive"]


def load_phrasebank():
    """Load a script-free Financial PhraseBank (datasets v5 compatible)."""
    try:
        return load_dataset(PRIMARY_SOURCE)
    except Exception as exc:
        print(f"Primary source failed ({type(exc).__name__}); falling back to mirror.")
        ds = load_dataset(FALLBACK_SOURCE)
        # Normalize plain int labels to a ClassLabel so id<->name mapping works downstream.
        if not isinstance(ds["train"].features["label"], ClassLabel):
            ds = ds.cast_column("label", ClassLabel(names=LABEL_NAMES_FALLBACK))
        return ds


raw = load_phrasebank()
raw

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3100
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 776
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 970
    })
})

### 1a. Inspect the dataset

Always look at the data before touching the model: schema, label meanings, a few raw
rows, and the class balance.

In [4]:
train = raw["train"]

# Schema + label definitions (ClassLabel maps int id -> human name).
label_feature = train.features["label"]
LABEL_NAMES = label_feature.names
ID2LABEL = {i: name for i, name in enumerate(LABEL_NAMES)}
LABEL2ID = {name: i for i, name in ID2LABEL.items()}

print("Features   :", train.features)
print("Num rows   :", train.num_rows)
print("Label names:", LABEL_NAMES)
print("id2label   :", ID2LABEL)

Features   : {'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'neutral', 'positive'])}
Num rows   : 3100
Label names: ['negative', 'neutral', 'positive']
id2label   : {0: 'negative', 1: 'neutral', 2: 'positive'}


In [5]:
# A few sample rows with decoded labels.
sample = train.shuffle(seed=42).select(range(8))
sample_df = pd.DataFrame({
    "sentence": sample["sentence"],
    "label_id": sample["label"],
    "label": [ID2LABEL[i] for i in sample["label"]],
})
sample_df

,sentence,label_id,label
0,cents Scout for potential acquisition targets ...,1,neutral
1,"However , the brokers ' ratings on the stock d...",1,neutral
2,Investment management and investment advisory ...,1,neutral
3,As a result of these negotiations the company ...,0,negative
4,The company reported net sales of 302 mln euro...,1,neutral
5,The total investment in the company will be EU...,1,neutral
6,Pretax profit rose to EUR 0.6 mn from EUR 0.4 ...,2,positive
7,The total headcount reduction will be 50 perso...,0,negative


In [6]:
# Class balance — Financial PhraseBank is imbalanced (neutral dominates).
# This is why we'll track macro-F1, not just accuracy, when training.
counts = pd.Series(train["label"]).map(ID2LABEL).value_counts()
balance = pd.DataFrame({
    "count": counts,
    "pct": (counts / counts.sum() * 100).round(1),
})
print(balance)

# Token-length sense check (rough, by whitespace) to pick a sane max_length later.
word_lens = pd.Series([len(s.split()) for s in train["sentence"]])
print("\nWords per sentence — describe():")
print(word_lens.describe()[["mean", "50%", "max"]].round(1))

          count   pct
neutral    1852  59.7
positive    866  27.9
negative    382  12.3

Words per sentence — describe():
mean    23.1
50%     21.0
max     81.0
dtype: float64


## 2. Load the model + tokenizer

Start with `distilbert-base-uncased` (small, fast baseline). We pass the label maps so
the model config carries human-readable labels — handy for inference and debugging.

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(DEVICE)

# The classification head is freshly initialized (warning about this is EXPECTED):
# we're about to fine-tune it. The base encoder weights are pretrained.
print(type(model).__name__, "loaded on", DEVICE)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification loaded on mps


In [8]:
# Inspect the model: size, head shape, and config labels.
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {n_params:,}")
print(f"Trainable params: {n_trainable:,}")
print("Output labels   :", model.config.id2label)
print("Max input len   :", tokenizer.model_max_length)
print("\nClassifier head :", model.classifier)

Total params    : 66,955,779
Trainable params: 66,955,779
Output labels   : {0: 'negative', 1: 'neutral', 2: 'positive'}
Max input len   : 512

Classifier head : Linear(in_features=768, out_features=3, bias=True)


## 3. Wiring sanity check — tokenize + one forward pass

Run a single untrained forward pass to confirm tokenizer → model → logits → predicted
label all connect. Predictions will be random/garbage (the head is untrained) — that's
fine; we only care that shapes and the pipeline work.

In [9]:
examples = [
    "The company reported record quarterly profit and raised its dividend.",
    "Shares plunged after the firm warned of widening losses and layoffs.",
    "The board will meet on Thursday to review the quarterly filing.",
]

enc = tokenizer(examples, padding=True, truncation=True, max_length=128, return_tensors="pt")
print("input_ids shape:", enc["input_ids"].shape)  # (batch, seq_len)

model.eval()
with torch.no_grad():
    logits = model(**{k: v.to(DEVICE) for k, v in enc.items()}).logits

probs = torch.softmax(logits, dim=-1).cpu()
pred_ids = probs.argmax(dim=-1).tolist()

pd.DataFrame({
    "sentence": examples,
    "pred": [ID2LABEL[i] for i in pred_ids],
    **{f"p({name})": probs[:, i].round(decimals=3).tolist() for i, name in ID2LABEL.items()},
})

input_ids shape: torch.Size([3, 15])


,sentence,pred,p(negative),p(neutral),p(positive)
0,The company reported record quarterly profit a...,negative,0.346,0.345,0.310
1,Shares plunged after the firm warned of wideni...,negative,0.364,0.335,0.301
2,The board will meet on Thursday to review the ...,negative,0.361,0.341,0.298


## 4. Tokenize the full dataset (card 1.2)

Map the tokenizer over **all splits**. We pad/truncate to `max_length=128` — the
inspection above showed max ~81 words, so 128 subword tokens is plenty and keeps
batches uniform (fixed-length tensors).

In [10]:
MAX_LENGTH = 128


def tokenize_batch(batch: dict) -> dict:
    """Tokenize a batch of sentences; labels pass through unchanged."""
    return tokenizer(
        batch["sentence"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )


tokenized = raw.map(tokenize_batch, batched=True, remove_columns=["sentence"])
print(tokenized)
print("\nColumns:", tokenized["train"].column_names)
print("Train example keys:", list(tokenized["train"][0].keys()))
print("input_ids length:", len(tokenized["train"][0]["input_ids"]))

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3100
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 776
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 970
    })
})

Columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
Train example keys: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
input_ids length: 128


## 5. Trainer workflow — Iteration 2 (cards 2.1–2.3)

**Splits:** we use the pre-defined train/val/test from `atrost/financial_phrasebank`
(3100 / 776 / 970 = `sentences_50agree`). No re-split — keeps results comparable.

**Metrics:** accuracy **and macro-F1** (neutral dominates → accuracy alone is misleading).

**Training:** 3 epochs, `lr=2e-5`, `batch=16`, evaluate each epoch, keep the checkpoint
with the best **validation macro-F1** (`load_best_model_at_end=True`).

In [11]:
from pathlib import Path

import evaluate
from transformers import Trainer, TrainingArguments

NUM_EPOCHS = 3

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    """Accuracy + macro-F1 — F1 is the primary metric on this imbalanced dataset."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")
    return {**acc, **f1}


OUTPUT_DIR = Path("../data/models/phrasebank_distilbert_best")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=50,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print("\n=== Training complete ===")
print(f"Train loss: {train_result.training_loss:.4f}")
print(f"Train runtime: {train_result.metrics['train_runtime']:.1f}s")

# Persist best-checkpoint model + metrics for the Streamlit Sentiment Lab tab.
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
metrics = {
    "model_name": MODEL_NAME,
    "dataset": PRIMARY_SOURCE,
    "split_source": "atrost/financial_phrasebank pre-defined splits (sentences_50agree; no re-split)",
    "epochs": NUM_EPOCHS,
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 16,
    "max_length": MAX_LENGTH,
    "metric_for_best_model": "f1",
    "train_loss": float(train_result.training_loss),
    "train_runtime_s": float(train_result.metrics.get("train_runtime", 0)),
    "device": DEVICE,
}
metrics_path = OUTPUT_DIR / "metrics.json"

/Users/armandoordoricadelatorre/miniconda/envs/sentiment-ltr-paper/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.601167,0.534999,0.795103



=== Training complete ===
Train loss: 0.6874
Train runtime: 43.9s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
import json

val_metrics = trainer.evaluate(tokenized["validation"])
test_metrics = trainer.evaluate(tokenized["test"])
metrics["validation"] = {k: float(v) if isinstance(v, (int, float)) else v for k, v in val_metrics.items()}
metrics["test"] = {k: float(v) if isinstance(v, (int, float)) else v for k, v in test_metrics.items()}
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
print(f"Saved model + metrics to {OUTPUT_DIR.resolve()}")

print("Validation:", {k: round(v, 4) if isinstance(v, float) else v for k, v in val_metrics.items()})
print("Test      :", {k: round(v, 4) if isinstance(v, float) else v for k, v in test_metrics.items()})

# Post fine-tune: predictions on the three example sentences should look more sensible.
model = trainer.model
model.to(DEVICE)
model.eval()
enc = tokenizer(examples, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
with torch.no_grad():
    logits = model(**{k: v.to(DEVICE) for k, v in enc.items()}).logits
probs = torch.softmax(logits, dim=-1).cpu()
pred_ids = probs.argmax(dim=-1).tolist()

pd.DataFrame({
    "sentence": examples,
    "pred": [ID2LABEL[i] for i in pred_ids],
    **{f"p({name})": probs[:, i].round(decimals=3).tolist() for i, name in ID2LABEL.items()},
})

/Users/armandoordoricadelatorre/miniconda/envs/sentiment-ltr-paper/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.601167,0.534999,1,0.795103


/Users/armandoordoricadelatorre/miniconda/envs/sentiment-ltr-paper/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.601167,0.555196,1,0.784536


Saved model + metrics to /Users/armandoordoricadelatorre/Documents/U of T/PhD/PhD Research/Sentiment_learn_to_rank_paper/data/models/phrasebank_distilbert_1ep
Validation: {'eval_loss': 0.535, 'eval_accuracy': 0.7951}
Test      : {'eval_loss': 0.5552, 'eval_accuracy': 0.7845}


,sentence,pred,p(negative),p(neutral),p(positive)
0,The company reported record quarterly profit a...,neutral,0.049,0.610,0.341
1,Shares plunged after the firm warned of wideni...,negative,0.613,0.135,0.252
2,The board will meet on Thursday to review the ...,neutral,0.026,0.923,0.052
